# 🧠 Faithful by Design: XM-CBM Multimodal Clinical AI
## Complete Experimental Pipeline

**Paper**: *Faithful by Design: A Cross-Modal Concept Bottleneck Framework for Trustworthy Multimodal Clinical Decision Support*

**Runtime**: GPU (T4 or better) — Go to `Runtime → Change runtime type → T4 GPU`

This notebook runs the entire pipeline:
1. Clone repo & install dependencies
2. Generate synthetic MIMIC-IV + CXR data
3. Train all 7 baseline + proposed models
4. Compute predictive + faithfulness + stability metrics
5. Run ablation study
6. Generate publication-ready results tables

---
## 1. Setup & Installation

In [ ]:
# Clone the repository
!git clone https://github.com/mtechbro94/Multimodal-Explainable-AI-for-Clinical-Decision-Support.git
%cd Multimodal-Explainable-AI-for-Clinical-Decision-Support

# Install dependencies
!pip install -q torch torchvision timm xgboost lightgbm shap captum scikit-learn matplotlib seaborn tqdm tabulate

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 2. Import Project Modules

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import random
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, 'src')

from data.synthetic_mimic import SyntheticMIMICDataset
from models.tabular import TabularMLP, XGBoostWrapper, LightGBMWrapper
from models.imaging import DenseNet121Classifier, ViTClassifier
from models.fusion import LateFusionModel
from models.proposed import CrossModalConceptBottleneck, XMCBMLoss
from explainability.gradcam import GradCAM
from explainability.integrated_gradients import IntegratedGradients
from explainability.shap_explainers import TreeSHAPExplainer, KernelSHAPExplainer
from explainability.intrinsic import IntrinsicConceptExplainer
from evaluation.metrics import compute_auroc, compute_auprc, compute_brier_score, compute_ece, compute_all_metrics
from evaluation.faithfulness import compute_deletion_auc, compute_insertion_auc, compute_infidelity
from evaluation.stability import estimate_lipschitz_constant, explanation_sensitivity, topk_stability

print("✅ All modules imported successfully!")

---
## 3. Configuration & Seed

In [ ]:
# ==================== CONFIGURATION ====================
N_SAMPLES = 3000       # Number of synthetic patients
BATCH_SIZE = 64        # Training batch size
EPOCHS = 15            # Training epochs (reduce for faster demo)
LR = 1e-4              # Learning rate
N_FOLDS = 3            # Cross-validation folds (use 3 for speed, 5 for paper)
PATIENCE = 5           # Early stopping patience
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"Config: N={N_SAMPLES}, Epochs={EPOCHS}, Batch={BATCH_SIZE}, Folds={N_FOLDS}")

---
## 4. Generate Synthetic MIMIC Dataset

In [ ]:
dataset = SyntheticMIMICDataset(n_samples=N_SAMPLES, seed=SEED)

# Dataset statistics
labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])
print(f"\n📊 Dataset Statistics:")
print(f"  Total patients: {len(dataset)}")
print(f"  Tabular features: {dataset.get_tabular_dim()}")
print(f"  Feature names: {dataset.get_feature_names()[:10]}... ({len(dataset.get_feature_names())} total)")
print(f"  Image shape: {dataset[0]['image'].shape}")
print(f"  Mortality rate: {labels.mean():.1%} ({int(labels.sum())}/{len(labels)})")

# Quick visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sample positive and negative images
pos_idx = np.where(labels == 1)[0][0]
neg_idx = np.where(labels == 0)[0][0]

img_pos = dataset[pos_idx]['image'].permute(1, 2, 0).numpy()
img_neg = dataset[neg_idx]['image'].permute(1, 2, 0).numpy()
# Denormalize for visualization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img_pos = np.clip(img_pos * std + mean, 0, 1)
img_neg = np.clip(img_neg * std + mean, 0, 1)

axes[0].imshow(img_neg, cmap='gray')
axes[0].set_title('Survivor (Label=0)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_pos, cmap='gray')
axes[1].set_title('Non-Survivor (Label=1)', fontsize=12)
axes[1].axis('off')

# Feature distribution
tab_data = np.stack([dataset[i]['tabular'].numpy() for i in range(min(500, len(dataset)))])
features = dataset.get_feature_names()
axes[2].barh(features[:10], np.abs(tab_data[:, :10]).mean(axis=0))
axes[2].set_title('Mean Feature Magnitudes (first 10)', fontsize=12)
axes[2].set_xlabel('Mean |value|')

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dataset generated and visualized")

---
## 5. Training Helper Functions

In [ ]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.notebook import tqdm
import torch.nn as nn
import torch.optim as optim

def collate_fn(batch):
    return {
        'tabular': torch.stack([b['tabular'] for b in batch]),
        'image': torch.stack([b['image'] for b in batch]),
        'label': torch.stack([b['label'] for b in batch])
    }

def train_epoch(model, loader, optimizer, criterion, device, model_type, loss_fn=None):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        targets = batch['label'].to(device)
        x_tab = batch['tabular'].to(device)
        x_img = batch['image'].to(device)

        if model_type == 'xm_cbm':
            logits, concept_dict = model(x_tab, x_img)
            loss, _ = loss_fn(logits, targets, concept_dict, model, x_tab, x_img)
        elif model_type in ['late_fusion']:
            logits = model(x_tab, x_img)
            loss = criterion(logits.view(-1), targets.view(-1))
        elif model_type in ['densenet', 'vit']:
            logits = model(x_img)
            loss = criterion(logits.view(-1), targets.view(-1))
        else:  # tabular models
            logits = model(x_tab)
            loss = criterion(logits.view(-1), targets.view(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate_model(model, loader, device, model_type):
    model.eval()
    y_true, y_prob = [], []
    for batch in loader:
        targets = batch['label']
        x_tab = batch['tabular'].to(device)
        x_img = batch['image'].to(device)

        if model_type == 'xm_cbm':
            logits, _ = model(x_tab, x_img)
        elif model_type in ['late_fusion']:
            logits = model(x_tab, x_img)
        elif model_type in ['densenet', 'vit']:
            logits = model(x_img)
        else:
            logits = model(x_tab)

        probs = torch.sigmoid(logits).cpu()
        y_true.extend(targets.numpy().flatten())
        y_prob.extend(probs.numpy().flatten())
    return np.array(y_true), np.array(y_prob)

def train_neural(model, train_loader, val_loader, model_type, epochs=EPOCHS, lr=LR, loss_fn=None):
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()
    best_auroc = -1
    best_state = None
    patience_counter = 0

    for ep in range(epochs):
        loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, model_type, loss_fn)
        scheduler.step()
        y_true, y_prob = evaluate_model(model, val_loader, DEVICE, model_type)
        auroc = roc_auc_score(y_true, y_prob)

        if auroc > best_auroc:
            best_auroc = auroc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    if best_state:
        model.load_state_dict(best_state)
        model.to(DEVICE)
    return best_auroc

print("✅ Training helpers defined")

---
## 6. Train All Models (K-Fold Cross-Validation)

In [ ]:
tab_dim = dataset.get_tabular_dim()
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Model configurations
NEURAL_MODELS = {
    'MLP': ('mlp', lambda: TabularMLP(input_dim=tab_dim).to(DEVICE)),
    'DenseNet-121': ('densenet', lambda: DenseNet121Classifier().to(DEVICE)),
    'ViT-B/16': ('vit', lambda: ViTClassifier().to(DEVICE)),
    'Late Fusion': ('late_fusion', lambda: LateFusionModel(
        TabularMLP(input_dim=tab_dim), DenseNet121Classifier()).to(DEVICE)),
    'XM-CBM (Ours)': ('xm_cbm', lambda: CrossModalConceptBottleneck(tab_input_dim=tab_dim).to(DEVICE)),
}

all_results = {}

# ===== TREE-BASED MODELS =====
print("="*60)
print("🌲 Training Tree-Based Models")
print("="*60)

for name, ModelClass in [('XGBoost', XGBoostWrapper), ('LightGBM', LightGBMWrapper)]:
    fold_results = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        X_train = np.stack([dataset[i]['tabular'].numpy() for i in train_idx])
        y_train = np.array([dataset[i]['label'].item() for i in train_idx])
        X_val = np.stack([dataset[i]['tabular'].numpy() for i in val_idx])
        y_val = np.array([dataset[i]['label'].item() for i in val_idx])

        model = ModelClass()
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
        y_prob = model.predict_proba(X_val)
        metrics = compute_all_metrics(y_val, y_prob)
        fold_results.append(metrics)

    all_results[name] = fold_results
    avg = {k: np.mean([f[k] for f in fold_results]) for k in fold_results[0]}
    print(f"  {name}: AUROC={avg['auroc']:.4f}, AUPRC={avg['auprc']:.4f}")

# ===== NEURAL MODELS =====
print(f"\n{'='*60}")
print("🧠 Training Neural Models")
print("="*60)

trained_models = {}  # Store best models for later explanation

for name, (model_type, constructor) in NEURAL_MODELS.items():
    print(f"\n▶ Training {name}...")
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        set_seed(SEED + fold)
        model = constructor()
        loss_fn = XMCBMLoss() if model_type == 'xm_cbm' else None

        train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE,
                                 shuffle=True, collate_fn=collate_fn, num_workers=0)
        val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE,
                               shuffle=False, collate_fn=collate_fn, num_workers=0)

        best_auroc = train_neural(model, train_loader, val_loader, model_type, loss_fn=loss_fn)

        y_true, y_prob = evaluate_model(model, val_loader, DEVICE, model_type)
        metrics = compute_all_metrics(y_true, y_prob)
        fold_results.append(metrics)
        print(f"    Fold {fold+1}/{N_FOLDS}: AUROC={metrics['auroc']:.4f}")

    all_results[name] = fold_results
    trained_models[name] = model  # Keep last fold model
    avg = {k: np.mean([f[k] for f in fold_results]) for k in fold_results[0]}
    print(f"  ✓ {name}: AUROC={avg['auroc']:.4f}±{np.std([f['auroc'] for f in fold_results]):.4f}")

print(f"\n{'='*60}")
print("✅ All models trained!")
print("="*60)

---
## 7. Generate Explanations & Compute Faithfulness

In [ ]:
print("🔍 Computing Explanations & Faithfulness Metrics...")
print("(This may take a few minutes)\n")

N_EVAL_SAMPLES = 30  # Number of samples for faithfulness evaluation

# Get sample data
sample_indices = np.random.choice(len(dataset), N_EVAL_SAMPLES, replace=False)
sample_tab = torch.stack([dataset[i]['tabular'] for i in sample_indices]).to(DEVICE)
sample_img = torch.stack([dataset[i]['image'] for i in sample_indices]).to(DEVICE)
sample_labels = np.array([dataset[i]['label'].item() for i in sample_indices])

faithfulness_results = {}

# === XM-CBM Intrinsic ===
if 'XM-CBM (Ours)' in trained_models:
    print("▶ XM-CBM Intrinsic Attribution...")
    xm_cbm = trained_models['XM-CBM (Ours)']
    xm_cbm.eval()
    explainer = IntrinsicConceptExplainer(xm_cbm)

    del_aucs, ins_aucs, infids = [], [], []
    for i in range(N_EVAL_SAMPLES):
        try:
            x_t = sample_tab[i:i+1]
            x_i = sample_img[i:i+1]
            attr = explainer.explain(x_t, x_i)

            # Model function for faithfulness
            def model_fn(x):
                with torch.no_grad():
                    if not torch.is_tensor(x):
                        x = torch.tensor(x, dtype=torch.float32).to(DEVICE)
                    logits, _ = xm_cbm(x, x_i.expand(x.shape[0], -1, -1, -1))
                    return torch.sigmoid(logits).cpu().numpy().flatten()

            # Use tabular attributions for faithfulness
            if 'tab_concept_attributions' in attr:
                tab_attr = attr['tab_concept_attributions'].detach().cpu()
            else:
                tab_attr = torch.ones(x_t.shape[-1])

            d = compute_deletion_auc(model_fn, x_t.cpu(), tab_attr, n_steps=10)
            ins = compute_insertion_auc(model_fn, x_t.cpu(), tab_attr, n_steps=10)
            inf = compute_infidelity(model_fn, x_t.cpu(), tab_attr, n_perturbations=20)
            del_aucs.append(d)
            ins_aucs.append(ins)
            infids.append(inf)
        except Exception as e:
            continue

    faithfulness_results['XM-CBM'] = {
        'del_auc': (np.mean(del_aucs), np.std(del_aucs)) if del_aucs else (0,0),
        'ins_auc': (np.mean(ins_aucs), np.std(ins_aucs)) if ins_aucs else (0,0),
        'infidelity': (np.mean(infids), np.std(infids)) if infids else (0,0),
    }
    print(f"  Del-AUC: {faithfulness_results['XM-CBM']['del_auc'][0]:.4f}")
    print(f"  Ins-AUC: {faithfulness_results['XM-CBM']['ins_auc'][0]:.4f}")
    print(f"  Infidelity: {faithfulness_results['XM-CBM']['infidelity'][0]:.4f}")

# === MLP + KernelSHAP ===
if 'MLP' in trained_models:
    print("\n▶ MLP KernelSHAP Attribution...")
    mlp = trained_models['MLP']
    mlp.eval()

    del_aucs, ins_aucs, infids = [], [], []
    for i in range(min(N_EVAL_SAMPLES, 15)):  # fewer for KernelSHAP (slow)
        try:
            x_t = sample_tab[i:i+1]
            def mlp_fn(x):
                with torch.no_grad():
                    if not torch.is_tensor(x):
                        x = torch.tensor(x, dtype=torch.float32).to(DEVICE)
                    return torch.sigmoid(mlp(x)).cpu().numpy().flatten()

            # Simple gradient attribution as proxy
            x_t_grad = x_t.clone().requires_grad_(True)
            out = mlp(x_t_grad)
            out.backward()
            attr = x_t_grad.grad.detach().cpu()

            d = compute_deletion_auc(mlp_fn, x_t.cpu(), attr, n_steps=10)
            ins = compute_insertion_auc(mlp_fn, x_t.cpu(), attr, n_steps=10)
            inf = compute_infidelity(mlp_fn, x_t.cpu(), attr, n_perturbations=20)
            del_aucs.append(d)
            ins_aucs.append(ins)
            infids.append(inf)
        except Exception:
            continue

    faithfulness_results['MLP'] = {
        'del_auc': (np.mean(del_aucs), np.std(del_aucs)) if del_aucs else (0,0),
        'ins_auc': (np.mean(ins_aucs), np.std(ins_aucs)) if ins_aucs else (0,0),
        'infidelity': (np.mean(infids), np.std(infids)) if infids else (0,0),
    }
    print(f"  Del-AUC: {faithfulness_results['MLP']['del_auc'][0]:.4f}")
    print(f"  Ins-AUC: {faithfulness_results['MLP']['ins_auc'][0]:.4f}")
    print(f"  Infidelity: {faithfulness_results['MLP']['infidelity'][0]:.4f}")

print("\n✅ Faithfulness evaluation complete")

---
## 8. 📊 Table 1: Main Benchmark Results

In [ ]:
from tabulate import tabulate

# Compile results
table_rows = []
modality_map = {
    'XGBoost': 'Tab', 'LightGBM': 'Tab', 'MLP': 'Tab',
    'DenseNet-121': 'Img', 'ViT-B/16': 'Img',
    'Late Fusion': 'Tab+Img', 'XM-CBM (Ours)': 'Tab+Img'
}

for name in ['XGBoost', 'LightGBM', 'MLP', 'DenseNet-121', 'ViT-B/16', 'Late Fusion', 'XM-CBM (Ours)']:
    if name not in all_results:
        continue
    folds = all_results[name]
    row = {
        'Model': f"**{name}**" if name == 'XM-CBM (Ours)' else name,
        'Modality': modality_map.get(name, ''),
        'AUROC': f"{np.mean([f['auroc'] for f in folds]):.3f}±{np.std([f['auroc'] for f in folds]):.3f}",
        'AUPRC': f"{np.mean([f['auprc'] for f in folds]):.3f}±{np.std([f['auprc'] for f in folds]):.3f}",
        'Brier': f"{np.mean([f['brier'] for f in folds]):.3f}±{np.std([f['brier'] for f in folds]):.3f}",
        'ECE': f"{np.mean([f['ece'] for f in folds]):.3f}±{np.std([f['ece'] for f in folds]):.3f}",
    }
    table_rows.append(row)

df_results = pd.DataFrame(table_rows)
print("\n" + "="*80)
print("  TABLE 1: Main Benchmark Results (K-Fold Cross-Validation)")
print("="*80)
print(tabulate(df_results, headers='keys', tablefmt='github', showindex=False))

# Save
os.makedirs('results', exist_ok=True)
df_results.to_csv('results/benchmark_results.csv', index=False)
print("\n📁 Saved to results/benchmark_results.csv")

---
## 9. Ablation Study

In [ ]:
print("🔬 Running Ablation Study...")
print("Training XM-CBM variants with different loss configurations\n")

ablation_configs = [
    ('Full XM-CBM', {'lambda_concept': 0.1, 'lambda_faith': 0.5, 'lambda_align': 0.2}),
    ('w/o L_faith', {'lambda_concept': 0.1, 'lambda_faith': 0.0, 'lambda_align': 0.2}),
    ('w/o L_align', {'lambda_concept': 0.1, 'lambda_faith': 0.5, 'lambda_align': 0.0}),
    ('w/o L_concept', {'lambda_concept': 0.0, 'lambda_faith': 0.5, 'lambda_align': 0.2}),
]

ablation_results = []

for config_name, loss_params in ablation_configs:
    print(f"  ▶ Training: {config_name}...")
    fold_aucs = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        set_seed(SEED + fold)
        model = CrossModalConceptBottleneck(tab_input_dim=tab_dim).to(DEVICE)
        loss_fn = XMCBMLoss(**loss_params)

        train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE,
                                 shuffle=True, collate_fn=collate_fn, num_workers=0)
        val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE,
                               shuffle=False, collate_fn=collate_fn, num_workers=0)

        train_neural(model, train_loader, val_loader, 'xm_cbm', epochs=10, loss_fn=loss_fn)
        y_true, y_prob = evaluate_model(model, val_loader, DEVICE, 'xm_cbm')
        fold_aucs.append(roc_auc_score(y_true, y_prob))

    ablation_results.append({
        'Configuration': config_name,
        'AUROC': f"{np.mean(fold_aucs):.3f}±{np.std(fold_aucs):.3f}",
    })
    print(f"    AUROC: {np.mean(fold_aucs):.4f}±{np.std(fold_aucs):.4f}")

df_ablation = pd.DataFrame(ablation_results)
print("\n" + "="*60)
print("  TABLE 2: Ablation Study")
print("="*60)
print(tabulate(df_ablation, headers='keys', tablefmt='github', showindex=False))
df_ablation.to_csv('results/ablation_results.csv', index=False)
print("\n✅ Ablation study complete")

---
## 10. 📈 Visualization: Publication-Ready Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plt.suptitle('XM-CBM: Experimental Results Overview', fontsize=16, fontweight='bold')

# ---- Plot 1: AUROC comparison ----
model_names = list(all_results.keys())
auroc_means = [np.mean([f['auroc'] for f in all_results[m]]) for m in model_names]
auroc_stds = [np.std([f['auroc'] for f in all_results[m]]) for m in model_names]
colors = ['#4472C4'] * len(model_names)
if 'XM-CBM (Ours)' in model_names:
    colors[model_names.index('XM-CBM (Ours)')] = '#E74C3C'

axes[0,0].barh(model_names, auroc_means, xerr=auroc_stds, color=colors, edgecolor='black', linewidth=0.5)
axes[0,0].set_xlabel('AUROC', fontsize=11)
axes[0,0].set_title('(a) Predictive Performance', fontsize=12, fontweight='bold')
axes[0,0].set_xlim(0.5, 1.0)

# ---- Plot 2: ECE comparison ----
ece_means = [np.mean([f['ece'] for f in all_results[m]]) for m in model_names]
axes[0,1].barh(model_names, ece_means, color=colors, edgecolor='black', linewidth=0.5)
axes[0,1].set_xlabel('ECE (lower is better)', fontsize=11)
axes[0,1].set_title('(b) Calibration Error', fontsize=12, fontweight='bold')

# ---- Plot 3: Faithfulness comparison ----
if faithfulness_results:
    faith_models = list(faithfulness_results.keys())
    metrics = ['del_auc', 'ins_auc', 'infidelity']
    x_pos = np.arange(len(metrics))
    width = 0.35

    for i, m in enumerate(faith_models):
        vals = [faithfulness_results[m][met][0] for met in metrics]
        c = '#E74C3C' if 'CBM' in m else '#4472C4'
        axes[1,0].bar(x_pos + i*width, vals, width, label=m, color=c, edgecolor='black', linewidth=0.5)

    axes[1,0].set_xticks(x_pos + width/2)
    axes[1,0].set_xticklabels(['Del-AUC↓', 'Ins-AUC↑', 'Infidelity↓'])
    axes[1,0].legend()
    axes[1,0].set_title('(c) Explanation Faithfulness', fontsize=12, fontweight='bold')
else:
    axes[1,0].text(0.5, 0.5, 'Faithfulness data\nnot computed', transform=axes[1,0].transAxes,
                   ha='center', va='center', fontsize=14)

# ---- Plot 4: XM-CBM Concept Activations ----
if 'XM-CBM (Ours)' in trained_models:
    xm_cbm = trained_models['XM-CBM (Ours)']
    xm_cbm.eval()
    with torch.no_grad():
        _, concepts = xm_cbm(sample_tab[:5], sample_img[:5])
    shared = concepts['shared_concepts'].cpu().numpy().mean(axis=0)
    concept_names = [f'C{i+1}' for i in range(len(shared))]
    colors_c = ['#E74C3C' if v > np.median(shared) else '#4472C4' for v in shared]
    axes[1,1].bar(concept_names, shared, color=colors_c, edgecolor='black', linewidth=0.5)
    axes[1,1].set_xlabel('Shared Concepts', fontsize=11)
    axes[1,1].set_ylabel('Mean Activation', fontsize=11)
    axes[1,1].set_title('(d) XM-CBM Concept Activations', fontsize=12, fontweight='bold')
else:
    axes[1,1].text(0.5, 0.5, 'XM-CBM not trained', transform=axes[1,1].transAxes,
                   ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.savefig('results/figure_main_results.png', dpi=300, bbox_inches='tight')
plt.show()
print("📁 Figure saved to results/figure_main_results.png")

---
## 11. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Get predictions from last fold for each neural model
val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE,
                       shuffle=False, collate_fn=collate_fn, num_workers=0)

color_map = {
    'MLP': '#2196F3', 'DenseNet-121': '#4CAF50', 'ViT-B/16': '#FF9800',
    'Late Fusion': '#9C27B0', 'XM-CBM (Ours)': '#E74C3C'
}

for name, (model_type, _) in NEURAL_MODELS.items():
    if name in trained_models:
        model = trained_models[name]
        y_true, y_prob = evaluate_model(model, val_loader, DEVICE, model_type)

        # ROC
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auroc = roc_auc_score(y_true, y_prob)
        lw = 3 if 'CBM' in name else 1.5
        ax1.plot(fpr, tpr, label=f'{name} ({auroc:.3f})',
                color=color_map.get(name, 'gray'), linewidth=lw)

        # PR
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
        ax2.plot(rec, prec, label=f'{name} ({auprc:.3f})',
                color=color_map.get(name, 'gray'), linewidth=lw)

ax1.plot([0,1], [0,1], 'k--', alpha=0.3)
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curves', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/figure_roc_pr_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("📁 Figure saved to results/figure_roc_pr_curves.png")

---
## 12. Export All Results

In [ ]:
# Compile everything into a JSON report
final_report = {
    'config': {
        'n_samples': N_SAMPLES, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'n_folds': N_FOLDS, 'lr': LR, 'seed': SEED,
        'device': str(DEVICE)
    },
    'predictive_results': {},
    'faithfulness_results': {}
}

for name, folds in all_results.items():
    final_report['predictive_results'][name] = {
        k: {'mean': float(np.mean([f[k] for f in folds])),
            'std': float(np.std([f[k] for f in folds]))}
        for k in folds[0]
    }

for name, metrics in faithfulness_results.items():
    final_report['faithfulness_results'][name] = {
        k: {'mean': float(v[0]), 'std': float(v[1])}
        for k, v in metrics.items()
    }

with open('results/full_experiment_report.json', 'w') as f:
    json.dump(final_report, f, indent=2)

print("\n" + "="*60)
print("  📊 EXPERIMENT COMPLETE — FINAL SUMMARY")
print("="*60)
print(f"\nDataset: {N_SAMPLES} synthetic patients ({int(labels.sum())} positive)")
print(f"Cross-validation: {N_FOLDS}-fold stratified")
print(f"Models trained: {len(all_results)}")
print(f"\n📁 Output files:")
print(f"  results/benchmark_results.csv")
print(f"  results/ablation_results.csv")
print(f"  results/full_experiment_report.json")
print(f"  results/figure_main_results.png")
print(f"  results/figure_roc_pr_curves.png")
print(f"\n✅ All done! Results ready for manuscript integration.")